# Traffic Data Processing and Aggregation

This notebook processes traffic data from various Excel files, cleans and merges them, and then aggregates the data by date to create a final dataset. The data includes speed and flow information for different highway segments.

## Download Data

This cell downloads the `PeMS.zip` file containing traffic data from a GitHub repository. The `wget` command is used to retrieve the file.

In [ ]:
!wget PeMS.zip https://github.com/hyrumcorey/Data-Engineering-Pipeline/raw/refs/heads/main/PeMS.zip

--2025-04-11 01:54:43--  http://pems.zip/
Resolving pems.zip (pems.zip)... failed: Name or service not known.
wget: unable to resolve host address ‘pems.zip’
--2025-04-11 01:54:43--  https://github.com/hyrumcorey/Data-Engineering-Pipeline/raw/refs/heads/main/PeMS.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/hyrumcorey/Data-Engineering-Pipeline/refs/heads/main/PeMS.zip [following]
--2025-04-11 01:54:44--  https://raw.githubusercontent.com/hyrumcorey/Data-Engineering-Pipeline/refs/heads/main/PeMS.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3318597 (3.2M) [application/zip]
Saving to:

## Unzip Data Files

This cell extracts the contents of the downloaded `PeMS.zip` file into a specified folder `/content/traffic_folder/`. The `-o` flag is used to overwrite existing files without prompting.

In [ ]:
!unzip -o PeMS.zip -d /content/traffic_folder/

Archive:  PeMS.zip
  inflating: /content/traffic_folder/I15_N_Bound_SLC_Speed.xlsx  
  inflating: /content/traffic_folder/I15_N-Bound_SLC_Flow.xlsx  
  inflating: /content/traffic_folder/I15_S-Bound_SLC_Flow.xlsx  
  inflating: /content/traffic_folder/I15_S-Bound_SLC_Speed.xlsx  
  inflating: /content/traffic_folder/StStreet_N_Bound_Y23-24.xlsx  
  inflating: /content/traffic_folder/StStreet_N_Bound_Y24-25.xlsx  
  inflating: /content/traffic_folder/StStreet_S_Bound_Y23-24.xlsx  
  inflating: /content/traffic_folder/StStreet_S_Bound_Y24-25.xlsx  


## Load Individual Excel Files (Initial Load)

This cell loads each of the individual Excel files (for I-15 and State Street traffic data) into separate pandas DataFrames. This is an initial loading step before concatenation.

In [ ]:
import pandas as pd

i15_nb_flow = pd.read_excel("/content/traffic_folder/I15_N-Bound_SLC_Flow.xlsx")
i15_nb_speed = pd.read_excel("/content/traffic_folder/I15_N_Bound_SLC_Speed.xlsx")
i15_sb_flow = pd.read_excel("/content/traffic_folder/I15_S-Bound_SLC_Flow.xlsx")
i15_sb_speed = pd.read_excel("/content/traffic_folder/I15_S-Bound_SLC_Speed.xlsx")
ss_nb_23 = pd.read_excel("/content/traffic_folder/StStreet_N_Bound_Y23-24.xlsx")
ss_nb_24 = pd.read_excel("/content/traffic_folder/StStreet_N_Bound_Y24-25.xlsx")
ss_sb_23 = pd.read_excel("/content/traffic_folder/StStreet_S_Bound_Y23-24.xlsx")
ss_sb_24 = pd.read_excel("/content/traffic_folder/StStreet_S_Bound_Y24-25.xlsx")



## Display Sample Data (I-15 Southbound Flow)

This cell displays the first few rows of the `i15_sb_flow` DataFrame to provide a quick look at its structure and content.

In [ ]:
i15_sb_flow

,Hour,3103240-ML,5600-ML,# Lane Points,% Observed
0,2023-04-01 00:00:00,176,126,48,100.0
1,2023-04-01 01:00:00,107,79,48,100.0
2,2023-04-01 02:00:00,77,67,48,100.0
3,2023-04-01 03:00:00,69,71,48,100.0
4,2023-04-01 04:00:00,80,79,48,100.0
...,...,...,...,...,...
17560,2025-04-01 19:00:00,414,386,48,100.0
17561,2025-04-01 20:00:00,315,327,48,100.0
17562,2025-04-01 21:00:00,300,209,48,100.0
17563,2025-04-01 22:00:00,186,194,48,100.0


## Consolidate All Excel Files into a Single List

This cell iterates through all Excel files in the specified `traffic_folder`, reads each into a pandas DataFrame, adds a 'data_set' column derived from the filename, and appends them to `traffic_list`.

In [ ]:
import os
folder_path = "/content/traffic_folder"
traffic_list = []
import re
import numpy as np

# loop through all files, store in list, concat to df
for traffic_file in os.listdir(folder_path):
  file_path = folder_path + '/' + traffic_file
  df = pd.read_excel(file_path)
  df["data_set"] = re.sub(r"\.[a-zA-Z]+$", "", traffic_file) #regex to remove filetype from name
  traffic_list.append(df)

#traffic_df = pd.concat(traffic_list, axis=1, ignore_index=False)




## Concatenate All DataFrames and Display Head

This cell concatenates all DataFrames from `traffic_list` into a single DataFrame `traffic_df`, stacking them vertically (`axis=0`). It then displays the first 3 rows of the merged DataFrame.

In [ ]:
traffic_df = pd.concat(traffic_list, axis=0, ignore_index=True)
traffic_df.head(3)

,Hour,3103240-ML,5600-ML,# Lane Points,% Observed,data_set,Lane 1 Flow (Veh/Hour),Lane 1 Speed (mph),Lane 2 Flow (Veh/Hour),Lane 2 Speed (mph),Lane 3 Flow (Veh/Hour),Lane 3 Speed (mph),Flow (Veh/Hour),Speed (mph),Lane 4 Flow (Veh/Hour),Lane 4 Speed (mph),3103241-ML,5601-ML
0,2023-04-01 00:00:00,176.0,126.0,48,100.0,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-04-01 01:00:00,107.0,79.0,48,100.0,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-04-01 02:00:00,77.0,67.0,48,100.0,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Calculate Average Northbound I-15 Speed

This cell calculates the average speed for northbound I-15 traffic based on relevant columns ('3103241-ML' and '5601-ML') only for rows identified as 'Speed' datasets, storing the result in a new column `nb_i15_avg_speed`.

In [ ]:
traffic_df["nb_i15_avg_speed"] = np.where(
    traffic_df["data_set"].str.endswith("Speed"),
    (traffic_df["3103241-ML"] + traffic_df["5601-ML"]) / 2,
    np.nan
  )

## Calculate Average Northbound I-15 Flow

This cell calculates the average flow for northbound I-15 traffic based on relevant columns ('3103241-ML' and '5601-ML') only for rows identified as 'Flow' datasets, storing the result in a new column `nb_i15_avg_flow`.

In [ ]:
traffic_df["nb_i15_avg_flow"] = np.where(
    traffic_df["data_set"].str.endswith("Flow"),
    (traffic_df["3103241-ML"] + traffic_df["5601-ML"]) / 2,
    np.nan
  )

## Calculate Average Southbound I-15 Speed

This cell calculates the average speed for southbound I-15 traffic based on relevant columns ('3103240-ML' and '5600-ML') only for rows identified as 'Speed' datasets, storing the result in a new column `sb_i15_avg_speed`.

In [ ]:
traffic_df["sb_i15_avg_speed"] = np.where(
    traffic_df["data_set"].str.endswith("Speed"),
    (traffic_df["3103240-ML"] + traffic_df["5600-ML"]) / 2,
    np.nan
  )

## Calculate Average Southbound I-15 Flow

This cell calculates the average flow for southbound I-15 traffic based on relevant columns ('3103240-ML' and '5600-ML') only for rows identified as 'Flow' datasets, storing the result in a new column `sb_i15_avg_flow`.

In [ ]:
traffic_df["sb_i15_avg_flow"] = np.where(
    traffic_df["data_set"].str.endswith("Flow"),
    (traffic_df["3103240-ML"] + traffic_df["5600-ML"]) / 2,
    np.nan
  )

## Verify Northbound I-15 Flow Data

This cell filters `traffic_df` to show only rows corresponding to 'I15_N-Bound_SLC_Flow' and displays the first 3 rows, along with the newly calculated average flow, to verify the operation.

In [ ]:
traffic_df[traffic_df["data_set"] == "I15_N-Bound_SLC_Flow"].head(3)

,Hour,3103240-ML,5600-ML,# Lane Points,% Observed,data_set,Lane 1 Flow (Veh/Hour),Lane 1 Speed (mph),Lane 2 Flow (Veh/Hour),Lane 2 Speed (mph),...,Flow (Veh/Hour),Speed (mph),Lane 4 Flow (Veh/Hour),Lane 4 Speed (mph),3103241-ML,5601-ML,nb_i15_avg_speed,nb_i15_avg_flow,sb_i15_avg_speed,sb_i15_avg_flow
87535,2023-04-01 00:00:00,NaN,NaN,48,100.0,I15_N-Bound_SLC_Flow,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,174.0,240.0,NaN,207.0,NaN,NaN
87536,2023-04-01 01:00:00,NaN,NaN,48,100.0,I15_N-Bound_SLC_Flow,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,101.0,143.0,NaN,122.0,NaN,NaN
87537,2023-04-01 02:00:00,NaN,NaN,48,100.0,I15_N-Bound_SLC_Flow,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,78.0,91.0,NaN,84.5,NaN,NaN


## Display Head of Combined Traffic DataFrame

This cell displays the first two rows of the `traffic_df` to quickly review the structure and the newly added average speed and flow columns.

In [ ]:
traffic_df.head(2)

,Hour,3103240-ML,5600-ML,# Lane Points,% Observed,data_set,Lane 1 Flow (Veh/Hour),Lane 1 Speed (mph),Lane 2 Flow (Veh/Hour),Lane 2 Speed (mph),...,Flow (Veh/Hour),Speed (mph),Lane 4 Flow (Veh/Hour),Lane 4 Speed (mph),3103241-ML,5601-ML,nb_i15_avg_speed,nb_i15_avg_flow,sb_i15_avg_speed,sb_i15_avg_flow
0,2023-04-01 00:00:00,176.0,126.0,48,100.0,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,151.0
1,2023-04-01 01:00:00,107.0,79.0,48,100.0,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.0


## Display DataFrame Information (Data Types and Non-Null Counts)

This cell provides a concise summary of the `traffic_df` DataFrame, including data types of each column, non-null values, and memory usage. This helps in identifying missing values and data type inconsistencies.

In [ ]:
traffic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105100 entries, 0 to 105099
Data columns (total 22 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Hour                    105100 non-null  datetime64[ns]
 1   3103240-ML              35130 non-null   float64       
 2   5600-ML                 35130 non-null   float64       
 3   # Lane Points           105100 non-null  int64         
 4   % Observed              105100 non-null  float64       
 5   data_set                105100 non-null  object        
 6   Lane 1 Flow (Veh/Hour)  34840 non-null   float64       
 7   Lane 1 Speed (mph)      34840 non-null   float64       
 8   Lane 2 Flow (Veh/Hour)  34840 non-null   float64       
 9   Lane 2 Speed (mph)      34840 non-null   float64       
 10  Lane 3 Flow (Veh/Hour)  34840 non-null   float64       
 11  Lane 3 Speed (mph)      34840 non-null   float64       
 12  Flow (Veh/Hour)         34840 

## Drop Redundant and Raw Data Columns

This cell removes a list of columns from the `traffic_df` that are either raw input data (like individual lane flows/speeds) or intermediate columns no longer needed after calculating averages, to clean up the DataFrame.

In [ ]:
traffic_df = traffic_df.drop(columns=traffic_df.loc[:, ["3103241-ML", "5601-ML", "3103240-ML", "5600-ML", "Lane 1 Flow (Veh/Hour)", "Lane 1 Speed (mph)", "Lane 2 Flow (Veh/Hour)", "Lane 2 Speed (mph)", "Lane 3 Flow (Veh/Hour)", "Lane 3 Speed (mph)", "Lane 4 Flow (Veh/Hour)", "Lane 4 Speed (mph)", "# Lane Points", "% Observed"]].columns)

## Display Head After Column Dropping

This cell displays the first 3 rows of the `traffic_df` after dropping the specified columns, confirming the successful removal of the unnecessary columns.

In [ ]:
traffic_df.head(3)

,Hour,data_set,Flow (Veh/Hour),Speed (mph),nb_i15_avg_speed,nb_i15_avg_flow,sb_i15_avg_speed,sb_i15_avg_flow
0,2023-04-01 00:00:00,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,151.0
1,2023-04-01 01:00:00,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,93.0
2,2023-04-01 02:00:00,I15_S-Bound_SLC_Flow,NaN,NaN,NaN,NaN,NaN,72.0


## Filter by Data Set: I15 Northbound Flow

This cell filters the `traffic_df` to show only the rows that correspond to the 'I15_N-Bound_SLC_Flow' dataset. This is for inspection purposes.

In [ ]:
traffic_df[traffic_df["data_set"] == "I15_N-Bound_SLC_Flow"]

,Hour,data_set,nb_i15_avg_speed,nb_i15_avg_flow,sb_i15_avg_speed,sb_i15_avg_flow
87535,2023-04-01 00:00:00,I15_N-Bound_SLC_Flow,NaN,207.0,NaN,NaN
87536,2023-04-01 01:00:00,I15_N-Bound_SLC_Flow,NaN,122.0,NaN,NaN
87537,2023-04-01 02:00:00,I15_N-Bound_SLC_Flow,NaN,84.5,NaN,NaN
87538,2023-04-01 03:00:00,I15_N-Bound_SLC_Flow,NaN,77.5,NaN,NaN
87539,2023-04-01 04:00:00,I15_N-Bound_SLC_Flow,NaN,78.0,NaN,NaN
...,...,...,...,...,...,...
105095,2025-04-01 19:00:00,I15_N-Bound_SLC_Flow,NaN,492.0,NaN,NaN
105096,2025-04-01 20:00:00,I15_N-Bound_SLC_Flow,NaN,363.5,NaN,NaN
105097,2025-04-01 21:00:00,I15_N-Bound_SLC_Flow,NaN,312.5,NaN,NaN
105098,2025-04-01 22:00:00,I15_N-Bound_SLC_Flow,NaN,241.5,NaN,NaN


## Display Unique Data Set Names

This cell shows all unique values present in the 'data_set' column of `traffic_df`. This helps confirm the different types of traffic data loaded.

In [ ]:
traffic_df["data_set"].unique()

array(['I15_S-Bound_SLC_Flow', 'StStreet_S_Bound_Y24-25',
       'StStreet_S_Bound_Y23-24', 'I15_S-Bound_SLC_Speed',
       'StStreet_N_Bound_Y24-25', 'I15_N_Bound_SLC_Speed',
       'StStreet_N_Bound_Y23-24', 'I15_N-Bound_SLC_Flow'], dtype=object)

## Create Separate DataFrames for Each Traffic Segment and Type

This cell filters the main `traffic_df` into several new DataFrames, each representing a specific traffic segment (I-15 or State Street) and data type (speed or flow) for easier individual processing.

In [ ]:
i15_nb_speed = traffic_df[traffic_df["data_set"] == "I15_N_Bound_SLC_Speed"]
i15_nb_flow = traffic_df[traffic_df["data_set"] == "I15_N-Bound_SLC_Flow"]
i15_sb_speed = traffic_df[traffic_df["data_set"] == "I15_S-Bound_SLC_Speed"]
i15_sb_flow = traffic_df[traffic_df["data_set"] == "I15_S-Bound_SLC_Flow"]
ss_nb_23 = traffic_df[traffic_df["data_set"] == "StStreet_N_Bound_Y23-24"]
ss_nb_24 = traffic_df[traffic_df["data_set"] == "StStreet_N_Bound_Y24-25"]
ss_sb_23 = traffic_df[traffic_df["data_set"] == "StStreet_S_Bound_Y23-24"]
ss_sb_24 = traffic_df[traffic_df["data_set"] == "StStreet_S_Bound_Y24-25"]

## Concatenate Northbound State Street Data (Y23-24 and Y24-25)

This cell combines the two DataFrames `ss_nb_23` and `ss_nb_24`, which represent northbound State Street traffic data for different periods, into a single DataFrame `ss_nb`.

In [ ]:
ss_nb = pd.concat([ss_nb_23, ss_nb_24], ignore_index=True)

## Display Combined Northbound State Street Data

This cell displays the full `ss_nb` DataFrame, which contains the concatenated northbound State Street traffic data.

In [ ]:
ss_nb

,Hour,data_set,Flow (Veh/Hour),Speed (mph),nb_i15_avg_speed,nb_i15_avg_flow,sb_i15_avg_speed,sb_i15_avg_flow
0,2023-04-01 00:00:00,StStreet_N_Bound_Y23-24,238.0,38.4,NaN,NaN,NaN,NaN
1,2023-04-01 01:00:00,StStreet_N_Bound_Y23-24,172.0,38.8,NaN,NaN,NaN,NaN
2,2023-04-01 02:00:00,StStreet_N_Bound_Y23-24,107.0,38.8,NaN,NaN,NaN,NaN
3,2023-04-01 03:00:00,StStreet_N_Bound_Y23-24,61.0,36.7,NaN,NaN,NaN,NaN
4,2023-04-01 04:00:00,StStreet_N_Bound_Y23-24,35.0,36.5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
17415,2025-03-31 06:00:00,StStreet_N_Bound_Y24-25,124.0,37.1,NaN,NaN,NaN,NaN
17416,2025-03-31 07:00:00,StStreet_N_Bound_Y24-25,281.0,35.7,NaN,NaN,NaN,NaN
17417,2025-03-31 08:00:00,StStreet_N_Bound_Y24-25,327.0,35.8,NaN,NaN,NaN,NaN
17418,2025-03-31 09:00:00,StStreet_N_Bound_Y24-25,337.0,36.4,NaN,NaN,NaN,NaN


## Select Relevant Columns for Northbound State Street Data

This cell selects only the 'Hour', 'Flow (Veh/Hour)', and 'Speed (mph)' columns from the `ss_nb` DataFrame, preparing it for merging and further calculations.

In [ ]:
ss_nb = ss_nb[["Hour", "Flow (Veh/Hour)", "Speed (mph)"]]

## Concatenate Southbound State Street Data (Y23-24 and Y24-25)

This cell combines the two DataFrames `ss_sb_23` and `ss_sb_24`, representing southbound State Street traffic data for different periods, into a single DataFrame `ss_sb`.

In [ ]:
ss_sb = pd.concat([ss_sb_23, ss_sb_24], ignore_index=True)

## Select Relevant Columns for Southbound State Street Data

This cell selects only the 'Hour', 'Flow (Veh/Hour)', and 'Speed (mph)' columns from the `ss_sb` DataFrame, preparing it for merging and further calculations.

In [ ]:
ss_sb = ss_sb[["Hour", "Flow (Veh/Hour)", "Speed (mph)"]]

## Merge Northbound and Southbound State Street Data

This cell merges the `ss_nb` and `ss_sb` DataFrames based on the 'Hour' column. A left merge (`how="left"`) is used to combine the northbound and southbound traffic data for State Street.

In [ ]:
ss = pd.merge(ss_nb, ss_sb, on="Hour", how="left")

## Calculate Average Flow and Speed for State Street

This cell calculates the average flow and average speed for State Street by combining the northbound and southbound values (identified by `_x` and `_y` suffixes after merging) and stores them in new columns `avg_flow` and `avg_speed`.

In [ ]:
ss["avg_flow"] = (ss["Flow (Veh/Hour)_x"] + ss["Flow (Veh/Hour)_y"]) / 2
ss["avg_speed"] = (ss["Speed (mph)_x"] + ss["Speed (mph)_y"]) /2

## Drop Original Flow and Speed Columns for State Street

This cell removes the original, unaggregated flow and speed columns (e.g., 'Flow (Veh/Hour)_x', 'Speed (mph)_y') from the `ss` DataFrame, keeping only the newly calculated average columns.

In [ ]:
ss = ss.drop(columns = ss.loc[:, "Flow (Veh/Hour)_x" : "Speed (mph)_y"])

## Merge Northbound I-15 Speed and Flow Data

This cell merges the `i15_nb_speed` and `i15_nb_flow` DataFrames based on the 'Hour' column, combining the speed and flow information for northbound I-15 traffic.

In [ ]:
i15_nb = pd.merge(i15_nb_speed, i15_nb_flow, on="Hour", how="left")

## Select Relevant Columns for Northbound I-15 Data

This cell selects the 'Hour', 'nb_i15_avg_speed_x', and 'nb_i15_avg_flow_y' columns from the merged `i15_nb` DataFrame. The suffixes `_x` and `_y` are from the previous merge operation.

In [ ]:
i15_nb = i15_nb[["Hour", "nb_i15_avg_speed_x", "nb_i15_avg_flow_y"]]

## Display Northbound I-15 Data

This cell displays the `i15_nb` DataFrame, showing the hour, average northbound speed, and average northbound flow for I-15.

In [ ]:
i15_nb

,Hour,nb_i15_avg_speed_x,nb_i15_avg_flow_y
0,2023-04-01 00:00:00,80.40,207.0
1,2023-04-01 01:00:00,78.20,122.0
2,2023-04-01 02:00:00,78.50,84.5
3,2023-04-01 03:00:00,79.20,77.5
4,2023-04-01 04:00:00,79.90,78.0
...,...,...,...
17560,2025-04-01 19:00:00,80.85,492.0
17561,2025-04-01 20:00:00,79.75,363.5
17562,2025-04-01 21:00:00,78.95,312.5
17563,2025-04-01 22:00:00,78.55,241.5


## Merge Southbound I-15 Speed and Flow Data

This cell merges the `i15_sb_speed` and `i15_sb_flow` DataFrames based on the 'Hour' column, combining the speed and flow information for southbound I-15 traffic.

In [ ]:
i15_sb = pd.merge(i15_sb_speed, i15_sb_flow, on="Hour", how="left")

## Select Relevant Columns for Southbound I-15 Data

This cell selects the 'Hour', 'sb_i15_avg_speed_x', and 'sb_i15_avg_flow_y' columns from the merged `i15_sb` DataFrame, preparing it for further merging and calculations.

In [ ]:
i15_sb = i15_sb[["Hour", "sb_i15_avg_speed_x", "sb_i15_avg_flow_y"]]

## Merge Northbound and Southbound I-15 Data

This cell merges the `i15_nb` and `i15_sb` DataFrames based on the 'Hour' column. This combines the processed northbound and southbound I-15 traffic data.

In [ ]:
i15 = pd.merge(i15_nb, i15_sb, on="Hour", how="left")

## Display Merged I-15 Data

This cell displays the `i15` DataFrame, which now contains combined northbound and southbound average speed and flow data for I-15.

In [ ]:
i15

,Hour,nb_i15_avg_speed_x,nb_i15_avg_flow_y,sb_i15_avg_speed_x,sb_i15_avg_flow_y,avg_speed
0,2023-04-01 00:00:00,80.40,207.0,80.60,151.0,80.500
1,2023-04-01 01:00:00,78.20,122.0,79.50,93.0,78.850
2,2023-04-01 02:00:00,78.50,84.5,77.45,72.0,77.975
3,2023-04-01 03:00:00,79.20,77.5,77.65,70.0,78.425
4,2023-04-01 04:00:00,79.90,78.0,78.50,79.5,79.200
...,...,...,...,...,...,...
17560,2025-04-01 19:00:00,80.85,492.0,80.25,400.0,80.550
17561,2025-04-01 20:00:00,79.75,363.5,78.90,321.0,79.325
17562,2025-04-01 21:00:00,78.95,312.5,78.70,254.5,78.825
17563,2025-04-01 22:00:00,78.55,241.5,78.00,190.0,78.275


## Calculate Overall Average Speed and Flow for I-15

This cell calculates the overall average speed and flow for I-15 by taking the average of the northbound and southbound average speeds and flows, respectively.

In [ ]:
i15["avg_speed"] = (i15['nb_i15_avg_speed_x'] + i15['sb_i15_avg_speed_x']) / 2
i15["avg_flow"] = (i15['nb_i15_avg_flow_y'] + i15['sb_i15_avg_flow_y']) / 2

## Select Final Columns for I-15 Data

This cell selects only the 'Hour', 'avg_speed', and 'avg_flow' columns from the `i15` DataFrame, simplifying it to the aggregated I-15 traffic information.

In [ ]:
i15 = i15[["Hour", "avg_speed", "avg_flow"]]

## Merge I-15 and State Street Data into Final DataFrame

This cell merges the processed `i15` and `ss` DataFrames based on the 'Hour' column, creating a `final_df` that contains aggregated traffic data from both highway segments.

In [ ]:
final_df = pd.merge(i15, ss, on="Hour", how="left")

## Display Processed I-15 Data

This cell displays the `i15` DataFrame, showing the final aggregated average speed and flow for I-15.

In [ ]:
i15

,Hour,avg_speed,avg_flow
0,2023-04-01 00:00:00,80.500,179.00
1,2023-04-01 01:00:00,78.850,107.50
2,2023-04-01 02:00:00,77.975,78.25
3,2023-04-01 03:00:00,78.425,73.75
4,2023-04-01 04:00:00,79.200,78.75
...,...,...,...
17560,2025-04-01 19:00:00,80.550,446.00
17561,2025-04-01 20:00:00,79.325,342.25
17562,2025-04-01 21:00:00,78.825,283.50
17563,2025-04-01 22:00:00,78.275,215.75


## Display Processed State Street Data

This cell displays the `ss` DataFrame, showing the final aggregated average speed and flow for State Street.

In [ ]:
ss

,Hour,avg_flow,avg_speed
0,2023-04-01 00:00:00,289.5,38.55
1,2023-04-01 01:00:00,264.0,38.80
2,2023-04-01 02:00:00,167.5,37.90
3,2023-04-01 03:00:00,75.5,37.05
4,2023-04-01 04:00:00,47.5,36.85
...,...,...,...
17415,2025-03-31 06:00:00,85.5,38.05
17416,2025-03-31 07:00:00,193.5,36.95
17417,2025-03-31 08:00:00,253.5,37.60
17418,2025-03-31 09:00:00,280.0,37.10


## Extract Date and Time from 'Hour' Column

This cell extracts the date and time components from the 'Hour' column in `final_df` and stores them in new columns named 'date' and 'time', respectively. This prepares the data for date-based aggregation.

In [ ]:
final_df.loc[:, "date"] = final_df["Hour"].dt.date
final_df.loc[:, "time"] = final_df["Hour"].dt.time

## Rename and Reorder Columns in Final DataFrame

This cell renames the columns of `final_df` for clarity and then reorders them to place 'date' at the beginning, followed by the speed and flow columns for I-15 and State Street.

In [ ]:
# rename
final_df.columns = ["Hour","i15_speed", "i15_flow", "st_st_flow", "st_st_speed", "date", "time"]
# re-arrange
final_df = final_df [["date","i15_speed", "i15_flow", "st_st_speed", "st_st_flow"]]

## Aggregate Final Data by Date

This cell groups the `final_df` by the 'date' column and calculates the mean of all other numerical columns for each date, creating a new DataFrame `df_final`.

In [ ]:
df_final = final_df.groupby("date").agg("mean")

## Save Aggregated Data to CSV

This cell writes the `df_final` DataFrame, which contains the aggregated traffic data by date, to a CSV file named `traffic_data.csv`.

In [ ]:
#write this df to a csv
df_final.to_csv("traffic_data.csv")